# 01 — Data Ingestion & Cleaning

This notebook demonstrates the ETL pipeline for geothermal well-log data.

**Objectives:**
- Load raw well data from CSV
- Standardize column names and units
- Handle missing values and outliers
- Export clean, analysis-ready dataset

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.etl_pipeline import GeothermalETL

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully')

## 1.1 Data Ingestion

Load the synthetic well-log dataset containing temperature measurements at various depths for 20 geothermal wells.

In [ ]:
# Initialize and run the ETL pipeline
etl = GeothermalETL('../data/sample_wells.csv')
raw_data = etl.ingest()

print(f'Dataset shape: {raw_data.shape}')
print(f'Wells: {raw_data["well_id"].nunique()}')
print(f'\nColumn types:\n{raw_data.dtypes}')
raw_data.head(10)

## 1.2 Data Quality Assessment

Before cleaning, let's understand the data quality issues.

In [ ]:
# Missing values summary
print('Missing Values:')
print(raw_data.isnull().sum())
print(f'\nTotal missing: {raw_data.isnull().sum().sum()}')
print(f'Missing rate: {raw_data.isnull().sum().sum() / raw_data.size * 100:.2f}%')

In [ ]:
# Distribution of key numeric columns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

raw_data['depth_m'].hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Depth Distribution (m)')
axes[0].set_xlabel('Depth (m)')

raw_data['temperature_c'].hist(ax=axes[1], bins=30, color='orangered', edgecolor='white')
axes[1].set_title('Temperature Distribution (°C)')
axes[1].set_xlabel('Temperature (°C)')

raw_data.groupby('well_id').size().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Measurements per Well')
axes[2].set_xlabel('Well ID')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../images/01_data_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 Run Full ETL Pipeline

The `run_pipeline()` method executes all cleaning steps in sequence.

In [ ]:
# Run the complete pipeline
clean_data = etl.run_pipeline(missing_strategy='interpolate')

print('\n--- Pipeline Summary ---')
for k, v in etl.get_summary().items():
    print(f'  {k}: {v}')

## 1.4 Post-Cleaning Validation

In [ ]:
# Verify no missing values remain
print('Missing values after cleaning:')
print(clean_data.isnull().sum())

# Check computed features
print(f'\nNew columns added: gradient_c_per_km, depth_category')
print(clean_data[['well_id', 'depth_m', 'temperature_c', 'gradient_c_per_km', 'depth_category']].head(10))

In [ ]:
# Lithology distribution
fig, ax = plt.subplots(figsize=(8, 4))
clean_data['lithology'].value_counts().plot(kind='barh', ax=ax, color='teal')
ax.set_title('Lithology Distribution Across All Wells')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('../images/01_lithology_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Export Clean Data

In [ ]:
# Export for use in subsequent notebooks
etl.export('../data/clean_wells.csv')
print('Clean dataset exported to data/clean_wells.csv')

---
**Next:** [02 — Well-Log Analysis](02_well_log_analysis.ipynb)